In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from astropy.io import fits

from joblib import Parallel, delayed


import glob
import sys
import requests
import time
import os
from tqdm import tqdm

## Loading in Catalogue

In [2]:
data_fold = 'C:/Users/oryan/Documents/mergers_in_desi/MiD-revamped/data'

In [3]:
df = pd.read_csv(f'{data_fold}/merged-data.csv', index_col = 0)

In [4]:
df_needed = df[['id_str', 'ra', 'dec', 'est_petro_th50']].set_index('id_str')

## Downloading Images

In [5]:
def get_fits(row):
    
    id_str = row[0]
    ra = row[1]
    dec = row[2]
    petro_r50 = row[3]
        
    im_size_arcsec = np.round(5 * (2 * petro_r50), 3)
    im_size = int(np.ceil(im_size_arcsec / 0.262))
    
    if im_size > 500:
        im_size = 500
    
    save_dir = f'E:/GZ-DESI/images-r50-recalib/{id_str}-cutout.fits'
    
    if os.path.exists(save_dir):
        return os.path.basename(save_dir)
    
    url = f'http://www.legacysurvey.org/viewer/fits-cutout?ra={ra}&dec={dec}&layer=ls-dr10&size={im_size}&pixscale=0.262&bands=grz'    
    
    for _ in range(5):
        try:
            r = requests.get(url)
        except:
            time.sleep(1)
            continue
        
        if r.status_code == 200:
            break
        else:
            time.sleep(1)
            
        if r.status_code == 500:
            print('Server error!')
            sys.exit()
    
    with open(save_dir, 'wb') as f:
        f.write(r.content)
    
    return os.path.basename(save_dir)

In [6]:
all_dict = df_needed.to_dict(orient = 'index')

In [7]:
len(all_dict)

137963

In [8]:
results = Parallel(n_jobs = 6)(delayed(get_fits)(i) for i in tqdm(zip(list(all_dict.keys()), list(df_needed['ra'].values), list(df_needed['dec'].values), list(df_needed['est_petro_th50'].values))))

50532it [5:23:18,  3.16it/s]

KeyboardInterrupt: 

C:\Users\oryan\AppData\Local\Continuum\anaconda3\lib\site-packages\joblib\_memmapping_reducer.py:608: UserWarning: Failed to delete temporary folder: C:\Users\oryan\AppData\Local\Temp\joblib_memmapping_folder_3304_b3eec72a97aa49458f1188119a461056_c94b56a832094ea59ab48af929aa3b57
  .format(pool_subfolder))
C:\Users\oryan\AppData\Local\Continuum\anaconda3\lib\site-packages\joblib\_memmapping_reducer.py:608: UserWarning: Failed to delete temporary folder: C:\Users\oryan\AppData\Local\Temp\joblib_memmapping_folder_3304_b3eec72a97aa49458f1188119a461056_e04dc7b6352942b4968dd8a3a972de99
  .format(pool_subfolder))


In [ ]:
files_dict = {}

In [8]:
done_files = list(files_dict.keys())
for i in tqdm(list(all_dict.keys())):
    if i in done_files:
        continue
    files_dict[i] = get_fits(i, all_dict[i]['ra'], all_dict[i]['dec'], all_dict[i]['est_petro_th50'])

  0%|          | 60/137963 [02:26<93:19:09,  2.44s/it] 


KeyboardInterrupt: 

## Checking Images

In [12]:
files = glob.glob('E:/GZ-DESI/images-r50-recalib/*.fits')

In [17]:
files[0]

'E:/GZ-DESI/images-r50-recalib\\388975_4015-cutout.fits'

In [ ]:
with fits.open(files[1]) as hdul:
    header = hdul[0].header
    data = hdul[0].data

In [ ]:
for i in files:
    with fits.open(i) as hdul:
        header = hdul[0].header
        data = hdul[0].data

    plt.figure(figsize = (8,8))
    plt.imshow(np.log10(data[1,:,:]), origin = 'lower')
    plt.show()
    
    time.sleep(4)
    plt.close()